# 1 - Instalação, Imports e Padronização do Prompt

In [4]:
!pip install -q -U transformers accelerate bitsandbytes google-genai rouge-score scikit-learn pandas tqdm

import os
import re
import json
import time
import torch
import pandas as pd
from tqdm.auto import tqdm
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
from google import genai
from google.genai import types
from sklearn.metrics import accuracy_score, f1_score
from rouge_score import rouge_scorer

# Autenticação
login(token=userdata.get('HF_TOKEN'))
gemini_client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

# Carregar dados de teste e lista oficial das 27 intenções
with open("data/test_set_300.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

with open("data/valid_intents.json", "r", encoding="utf-8") as f:
    valid_intents = json.load(f)

INTENTS_STR = ", ".join(valid_intents)
print(f"Carregadas {len(test_data)} mensagens de teste e {len(valid_intents)} intenções.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.1.1 which is incompatible.
Carregadas 300 mensagens de teste e 27 intenções.


# 2 - Prompts Padronizados e Função de Extração/Métricas

In [5]:
SYSTEM_PROMPT = f"""You are a customer support AI assistant.
Your task is to:
1. Classify the customer's message into EXACTLY ONE of the following 27 valid intents:
[{INTENTS_STR}]
2. Write a helpful, polite, and concise customer support response.

Respond ONLY with a valid JSON object using this exact format:
{{"intent": "<exact_intent_name>", "response": "<your_reply>"}}"""

FEW_SHOT_EXAMPLES = """
Example 1:
Customer: "I want to cancel my order #12345"
Output: {"intent": "cancel_order", "response": "I can certainly help you cancel your order. Please provide your order details so I can process the cancellation right away."}

Example 2:
Customer: "How do I get my money back for a returned item?"
Output: {"intent": "get_refund", "response": "I would be happy to assist you with your refund request. Let me check the status of your return and guide you through the refund process."}

Example 3:
Customer: "I need to update my shipping address"
Output: {"intent": "change_shipping_address", "response": "I can help you update your shipping address. Could you please share the new address and your order number?"}
"""

def parse_model_output(raw_text: str):
    """Extrai intent e response mesmo se o modelo envolver o JSON em markdown."""
    cleaned = re.sub(r"```json|```", "", raw_text).strip()
    try:
        data = json.loads(cleaned)
        intent = str(data.get("intent", "unknown")).strip().lower()
        response = str(data.get("response", "")).strip()
        return intent, response
    except Exception:
        # Fallback caso o modelo pequeno quebre o JSON
        found_intent = "unknown"
        for v_intent in valid_intents:
            if v_intent in cleaned.lower():
                found_intent = v_intent
                break
        return found_intent, cleaned

def evaluate_predictions(df_preds: pd.DataFrame, model_name: str, cost_per_1k_usd: float):
    y_true = df_preds["true_intent"].tolist()
    y_pred = df_preds["pred_intent"].tolist()

    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)

    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_scores = [
        scorer.score(ref, pred)["rougeL"].fmeasure
        for ref, pred in zip(df_preds["true_response"], df_preds["pred_response"])
    ]
    avg_rouge = sum(rouge_scores) / len(rouge_scores)
    avg_latency = df_preds["latency_sec"].mean()

    return {
        "Modelo": model_name,
        "Acurácia (%)": round(acc * 100, 2),
        "F1-Macro (%)": round(f1_macro * 100, 2),
        "ROUGE-L": round(avg_rouge, 4),
        "Latência Média (s)": round(avg_latency, 3),
        "Custo / 1k msgs (USD)": round(cost_per_1k_usd, 4)
    }

# 3 - Baseline 1 e 2 — Qwen2.5-1.5B-Instruct (Zero-Shot e Few-Shot)

In [3]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

def run_qwen_inference(dataset_list, use_few_shot=False):
    results = []
    sys_msg = SYSTEM_PROMPT + (f"\n\nHere are some examples:\n{FEW_SHOT_EXAMPLES}" if use_few_shot else "")

    for item in tqdm(dataset_list, desc=f"Qwen ({'Few-Shot' if use_few_shot else 'Zero-Shot'})"):
        messages = [
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": item["instruction"]}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to(qwen_model.device)

        t0 = time.perf_counter()
        with torch.no_grad():
            outputs = qwen_model.generate(
                **inputs,
                max_new_tokens=180,
                do_sample=False, # Temperatura 0 para determinismo na avaliação
                pad_token_id=tokenizer.eos_token_id
            )
        latency = time.perf_counter() - t0

        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        raw_out = tokenizer.decode(generated_ids, skip_special_tokens=True)
        pred_intent, pred_resp = parse_model_output(raw_out)

        results.append({
            "instruction": item["instruction"],
            "true_intent": item["intent"],
            "pred_intent": pred_intent,
            "true_response": item["response"],
            "pred_response": pred_resp,
            "latency_sec": latency
        })
    return pd.DataFrame(results)

df_qwen_zero = run_qwen_inference(test_data, use_few_shot=False)
df_qwen_few = run_qwen_inference(test_data, use_few_shot=True)

os.makedirs("results", exist_ok=True)
df_qwen_zero.to_csv("results/preds_qwen_zeroshot.csv", index=False)
df_qwen_few.to_csv("results/preds_qwen_fewshot.csv", index=False)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen (Zero-Shot):   0%|          | 0/300 [00:00<?, ?it/s]

Qwen (Few-Shot):   0%|          | 0/300 [00:00<?, ?it/s]

# 4 - Benchmark de Fronteira — Gemini 3.5 Flash via API

In [22]:
# ==============================================================================
# NOTA DE REPRODUTIBILIDADE (COTA GRATUITA GOOGLE AI STUDIO):
# Devido aos limites de requisições por minuto (RPM) e cota diária (RPD) da
# camada gratuita da API do Gemini, esta célula implementa salvamento incremental
# linha a linha em 'results/preds_gemini_flash.csv'. Caso a execução seja
# interrompida por limite de cota (erro 429/RESOURCE_EXHAUSTED), basta aguardar
# a janela de cota e executar novamente esta célula: ela retomará exatamente
# da próxima mensagem pendente, sem perda de dados nem distorção na latência.
# ==============================================================================

GEMINI_MODEL = "gemini-3.5-flash-lite"
OUTPUT_CSV = "results/preds_gemini_flash.csv"
os.makedirs("results", exist_ok=True)

# 1. Carregar progresso anterior, se existir
if os.path.exists(OUTPUT_CSV):
    df_existing = pd.read_csv(OUTPUT_CSV)
    df_existing = df_existing[~df_existing["pred_intent"].isin(["error", "unknown"])].copy()
    results = df_existing.to_dict("records")
    done_instructions = set(df_existing["instruction"].tolist())
    print(f"Retomando progresso: {len(results)}/{len(test_data)} mensagens já processadas.")
else:
    results = []
    done_instructions = set()

pending_items = [item for item in test_data if item["instruction"] not in done_instructions]
total_input_tokens = 0
total_output_tokens = 0

# 2. Inferência sequencial com medição individual de latência e checkpoint em disco
for item in tqdm(pending_items, desc=f"{GEMINI_MODEL} (API)"):
    t0 = time.perf_counter()
    try:
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=item["instruction"],
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=0.0,
                max_output_tokens=200,
                response_mime_type="application/json"
            )
        )
        latency = time.perf_counter() - t0
        raw_out = response.text or ""

        if response.usage_metadata:
            total_input_tokens += response.usage_metadata.prompt_token_count or 0
            total_output_tokens += response.usage_metadata.candidates_token_count or 0

    except Exception as e:
        print(f"\nExecução pausada por limite de cota/API: {e}")
        print(f"Progresso salvo ({len(results)}/{len(test_data)}). Execute a célula novamente para continuar.")
        break

    pred_intent, pred_resp = parse_model_output(raw_out)
    results.append({
        "instruction": item["instruction"],
        "true_intent": item["intent"],
        "pred_intent": pred_intent,
        "true_response": item["response"],
        "pred_response": pred_resp,
        "latency_sec": latency
    })

    # Checkpoint imediato após cada requisição concluída
    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
    time.sleep(2.1)

df_gemini = pd.DataFrame(results)
print(f"\nTotal consolidado para {GEMINI_MODEL}: {len(df_gemini)}/{len(test_data)} mensagens.")

Retomando progresso: 300/300 mensagens já processadas.


gemini-3.5-flash-lite (API): 0it [00:00, ?it/s]


Total consolidado para gemini-3.5-flash-lite: 300/300 mensagens.


# 5 - Consolidação da Tabela de Baselines

In [25]:
# 1. Carregar os resultados salvos das três avaliações de baseline
df_qwen_zero = pd.read_csv("results/preds_qwen_zeroshot.csv")
df_qwen_few = pd.read_csv("results/preds_qwen_fewshot.csv")
df_gemini = pd.read_csv("results/preds_gemini_flash.csv")

# 2. Estimativa de custo operacional por 1.000 mensagens (USD)
# - Qwen2.5-1.5B (Self-hosted em GPU T4 ~$0.35/h): (latência_média / 3600) * 0.35 * 1000
cost_qwen_zero = (df_qwen_zero["latency_sec"].mean() / 3600) * 0.35 * 1000
cost_qwen_few = (df_qwen_few["latency_sec"].mean() / 3600) * 0.35 * 1000

# - Gemini 3.5 Flash-Lite (Tabela comercial ~$0.10/1M tokens input, ~$0.40/1M tokens output)
if total_input_tokens == 0:
    total_input_tokens = len(df_gemini) * 180
    total_output_tokens = len(df_gemini) * 45

custo_total_gemini = (total_input_tokens / 1_000_000) * 0.10 + (total_output_tokens / 1_000_000) * 0.40
cost_gemini_1k = (custo_total_gemini / len(df_gemini)) * 1000

# 3. Montar e salvar a tabela consolidada de métricas
summary_baselines = pd.DataFrame([
    evaluate_predictions(df_qwen_zero, "Qwen2.5-1.5B (Zero-Shot)", cost_qwen_zero),
    evaluate_predictions(df_qwen_few, "Qwen2.5-1.5B (Few-Shot)", cost_qwen_few),
    evaluate_predictions(df_gemini, "Gemini 3.5 Flash-Lite (API)", cost_gemini_1k)
])

summary_baselines.to_csv("results/baselines_summary.csv", index=False)
print("=== RESUMO COMPARATIVO DOS BASELINES (300 MENSAGENS) ===\n")
print(summary_baselines.to_string(index=False))

=== RESUMO COMPARATIVO DOS BASELINES (300 MENSAGENS) ===

                     Modelo  Acurácia (%)  F1-Macro (%)  ROUGE-L  Latência Média (s)  Custo / 1k msgs (USD)
   Qwen2.5-1.5B (Zero-Shot)         57.33         43.21   0.2025               1.980                 0.1925
    Qwen2.5-1.5B (Few-Shot)         53.67         36.20   0.1914               1.927                 0.1874
Gemini 3.5 Flash-Lite (API)         92.00         88.73   0.2162               0.973                 0.0360


6 - Verificar modelos disponíveis

In [10]:
for m in gemini_client.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemini-4-hydrogen
models/gemini-4-neon
models/antigravity-preview-05-2026
models/antigravity-preview-09-2026
models/antigravity-preview-latest
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026
